### ***LOADING THE WINDOWED AUDIO***

In [ ]:
from pathlib import Path

SAMPLE_RATE: int = 16_000
N_MELS: int = 26
N_MFCC: int = 13
F_MIN: float = 0.0
F_MAX: float = SAMPLE_RATE / 2
ROLLOFF_PERCENT: float = 0.85

FRAME_SIZE: int = 400
N_FFT: int = FRAME_SIZE

BATCH_SIZE: int = 4_096
# arbitrary btw
# batch size is just to save on memory and space
# not necessary but Little Ms. LLM gave it so ok

NOTEBOOK_DIR: Path = Path.cwd()
DIR_REPOSITORY_ROOT: Path = NOTEBOOK_DIR
DIR_INPUT_FFT: Path = DIR_REPOSITORY_ROOT / "preprocessing-03-fft.npy"
DIR_OUTPUT_MFCC: Path = DIR_REPOSITORY_ROOT / "extracted-mfcc.npy"

# check if file eixists
if not DIR_INPUT_FFT.is_file():
    raise FileNotFoundError(
        f"Missing FFT data: {DIR_INPUT_FFT.resolve()}\n"
        "Run preprocessing-01-framing.ipynb, preprocessing-02-windowing.ipynb, "
        "and preprocessing-03-fft.ipynb first."
    )

In [ ]:
import numpy as np

INPUT_FFT:np.ndarray = np.load(DIR_INPUT_FFT.resolve(), allow_pickle=True)

### ***CHECKING THE WINDOWED FRAMES***

note : Only load trusted NumPy files with `allow_pickle=True`. A pickled object array can contain unsafe code.

In [ ]:
# checking the file contents
if INPUT_FFT.ndim != 2 or INPUT_FFT.shape[1] != 5:
    raise ValueError(f"Expected an FFT array with shape (number_of_frames, 5), got {INPUT_FFT.shape}.")
if len(INPUT_FFT) == 0:
    raise ValueError("The FFT input contains no frames.")
EXPECTED_BINS: int = N_FFT // 2 + 1
first_power = np.asarray(INPUT_FFT[0, 4])
if first_power.shape != (EXPECTED_BINS,):
    raise ValueError(
        f"Expected {EXPECTED_BINS} power bins for N_FFT={N_FFT}, got {first_power.shape}."
    )

n = 25
print("="*50, "SHAPE", f"\t{INPUT_FFT.shape=}", sep="\n")
print("SHAPE", f"\t{INPUT_FFT[n].shape=}", sep="\n")
print("="*50, "DIR", f"\t{INPUT_FFT[n][0]=}", sep="\n")
print("="*50, "FRAME#", f"\t{INPUT_FFT[n][1]=}", sep="\n")

print("="*50, "SHAPE", f"\t{INPUT_FFT[n][2].shape=}", sep="\n")
print("FRAME_DATA", f"\t{INPUT_FFT[n][2]=}", sep="\n")

### ***DEFINE OUTPUT COLUMNS***

In [ ]:
feature_columns: list[str] = [
    "zcr_mean",
    "centroid_mean",
    "rolloff_mean",
    "rms_mean",
]
for coefficient in range(1, N_MFCC + 1):
    feature_columns.extend(
        [f"mfcc{coefficient:02}_mean", f"mfcc{coefficient:02}_std"]
    )

output_columns: list[str] = ["dir", "frame #", *feature_columns]
print(f"Output column count: {len(output_columns)}")
print(output_columns)

### ***MEL FILTERBANK*** and helper function

In [ ]:
import librosa

MEL_FILTERBANK: np.ndarray = librosa.filters.mel(
    sr=SAMPLE_RATE,
    n_fft=N_FFT,
    n_mels=N_MELS,
    fmin=F_MIN,
    fmax=F_MAX,
    htk=True,
    norm=None,
    dtype=np.float32,
)

assert MEL_FILTERBANK.shape == (N_MELS, EXPECTED_BINS)
assert MEL_FILTERBANK.dtype == np.float32
assert np.isfinite(MEL_FILTERBANK).all()
assert (MEL_FILTERBANK >= 0).all()

print(f"Mel filterbank shape: {MEL_FILTERBANK.shape}")

In [ ]:
from scipy.fft import dct

LOG_FLOOR: float = np.finfo(np.float32).eps

def power_to_mfcc(power_spectrum: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    power = np.asarray(power_spectrum, dtype=np.float32)

    # checking data shape
    if power.shape != (EXPECTED_BINS,):
        raise ValueError(f"Expected power shape {(EXPECTED_BINS,)}, got {power.shape}.")
    if not np.isfinite(power).all() or (power < 0).any():
        raise ValueError("The power spectrum must contain finite, nonnegative values.")

    # uses the mel filterbank
    mel_energies = MEL_FILTERBANK @ power

    # np.maximum filters anything smallter than LOG_FLOOR for noise
    # np.log gets the log of the mel_energies
    log_mel_energies:np.ndarray = np.log(np.maximum(mel_energies, LOG_FLOOR))

    # the DCT(Discrete Cosine Transform) step
    coefficients:np.ndarray = dct(log_mel_energies, type=2, norm="ortho")[:N_MFCC]
    return coefficients.astype(np.float32), log_mel_energies.astype(np.float32)

### ***EXTRACT THE DATA***

In [ ]:
MFCC_FRAMES: np.ndarray = np.empty((len(INPUT_FFT), 3), dtype=object)
LOG_MEL_ENERGIES: np.ndarray = np.empty((len(INPUT_FFT), N_MELS), dtype=np.float32)

for row_index, row in enumerate(INPUT_FFT):
    source_file, frame_number, _, _, power_spectrum = row
    coefficients, log_mel_energies = power_to_mfcc(power_spectrum)
    MFCC_FRAMES[row_index] = [source_file, frame_number, coefficients]
    LOG_MEL_ENERGIES[row_index] = log_mel_energies

print(f"Extracted {N_MFCC} MFCCs from each of {len(MFCC_FRAMES)} frames")
print("Columns: source_file, frame_number, mfcc")